In [0]:
display(dbutils.fs.ls("s3://virginia-energy-data-2026/raw/"))

path,name,size,modificationTime
s3://virginia-energy-data-2026/raw/va_generation_2020_2025.json,va_generation_2020_2025.json,8234271,1789018592000


In [0]:
raw_path = "s3://virginia-energy-data-2026/raw/va_generation_2020_2025.json"

df_raw = spark.read.option("multiLine", "true").json(raw_path)

display(df_raw)

response


In [0]:
df_raw.printSchema()

root
 |-- response: struct (nullable = true)
 |    |-- data: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- fuelTypeDescription: string (nullable = true)
 |    |    |    |-- fueltypeid: string (nullable = true)
 |    |    |    |-- generation: string (nullable = true)
 |    |    |    |-- generation-units: string (nullable = true)
 |    |    |    |-- location: string (nullable = true)
 |    |    |    |-- period: string (nullable = true)
 |    |    |    |-- sectorDescription: string (nullable = true)
 |    |    |    |-- sectorid: string (nullable = true)
 |    |    |    |-- stateDescription: string (nullable = true)
 |    |-- dateFormat: string (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- frequency: string (nullable = true)
 |    |-- total: long (nullable = true)



In [0]:
bronze_df = df_raw.selectExpr("explode(response.data) as record").select("record.*")

display(bronze_df)

fuelTypeDescription,fueltypeid,generation,generation-units,location,period,sectorDescription,sectorid,stateDescription
all fuels,ALL,7068.27934,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
all renewables,AOR,242.04817,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
biomass,BIO,111.15839,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
bituminous coal and synthetic coal,BIS,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
bituminous coal,BIT,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
"coal, excluding waste coal",COL,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
all coal products,COW,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
distillate fuel oil,DFO,36.56516,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
estimated small scale solar photovoltaic,DPV,0,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia
fossil fuels,FOS,4727.84399,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia


In [0]:
print(f"Bronze records: {bronze_df.count()}")

Bronze records: 22610


# BRONZE INGESTION NOTE:
# The initial Bronze transformation attempted to use input_file_name()
# to capture the source file path as ingestion metadata. Databricks Unity
# Catalog does not support input_file_name() in this environment and
# returned a UC_COMMAND_NOT_SUPPORTED error, recommending _metadata.file_path.
#
# Rather than add unnecessary complexity for this ingestion step, I removed
# the source-file metadata column and retained the ingestion timestamp.
# The source file is already controlled by the S3 raw landing path, so the
# additional file-path column was not required for this pipeline.
#
# The Bronze DataFrame is rebuilt from df_raw here to ensure the failed
# transformation is not retained in the DataFrame lineage.

In [0]:
from pyspark.sql.functions import explode, current_timestamp

# Rebuild Bronze DataFrame from the raw JSON
bronze_df = (
    df_raw
    .selectExpr("explode(response.data) as record")
    .select("record.*")
    .withColumn("_ingested_at", current_timestamp())
)

display(bronze_df)

fuelTypeDescription,fueltypeid,generation,generation-units,location,period,sectorDescription,sectorid,stateDescription,_ingested_at
all fuels,ALL,7068.27934,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
all renewables,AOR,242.04817,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
biomass,BIO,111.15839,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
bituminous coal and synthetic coal,BIS,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
bituminous coal,BIT,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
"coal, excluding waste coal",COL,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
all coal products,COW,458.76735,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
distillate fuel oil,DFO,36.56516,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
estimated small scale solar photovoltaic,DPV,0,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z
fossil fuels,FOS,4727.84399,thousand megawatthours,VA,2025-12,Electric Utility,1,Virginia,2026-09-10T06:32:51.139Z


In [0]:
bronze_table = "virginia_energy_bronze"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print("Bronze table created successfully.")

Bronze table created successfully.


In [0]:
spark.sql("""
    SELECT COUNT(*) AS record_count
    FROM virginia_energy_bronze
""").show()

+------------+
|record_count|
+------------+
|       22610|
+------------+



In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT period) AS distinct_months,
        MIN(period) AS earliest_period,
        MAX(period) AS latest_period,
        COUNT(DISTINCT fueltypeid) AS distinct_fuel_types
    FROM virginia_energy_bronze
""").show()

+-------------+---------------+---------------+-------------+-------------------+
|total_records|distinct_months|earliest_period|latest_period|distinct_fuel_types|
+-------------+---------------+---------------+-------------+-------------------+
|        22610|             72|        2020-01|      2025-12|                 36|
+-------------+---------------+---------------+-------------+-------------------+



In [0]:
spark.sql("""
    SELECT
        fueltypeid,
        fuelTypeDescription,
        COUNT(*) AS record_count
    FROM virginia_energy_bronze
    GROUP BY fueltypeid, fuelTypeDescription
    ORDER BY fueltypeid
""").show(50, truncate=False)

+----------+----------------------------------------+------------+
|fueltypeid|fuelTypeDescription                     |record_count|
+----------+----------------------------------------+------------+
|ALL       |all fuels                               |936         |
|AOR       |all renewables                          |864         |
|BIO       |biomass                                 |864         |
|BIS       |bituminous coal and synthetic coal      |760         |
|BIT       |bituminous coal                         |760         |
|COL       |coal, excluding waste coal              |760         |
|COW       |all coal products                       |760         |
|DFO       |distillate fuel oil                     |888         |
|DPV       |estimated small scale solar photovoltaic|432         |
|FOS       |fossil fuels                            |888         |
|HPS       |hydro-electric pumped storage           |288         |
|HYC       |conventional hydroelectric              |598      